# Séance 7 · Exercices — Kaggle Titanic (1/2) : explorer, préparer, première soumission · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

**Niveau de la séance : ⭐⭐ Intermédiaire**

**Comment travailler** : lis l'énoncé, code dans la cellule `# À toi`, lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
Tout tourne dans **Google Colab** (rien à installer). Exécute chaque cellule avec `Maj + Entrée`, dans l'ordre : les exercices réutilisent ce qui précède.


## Quiz d'ouverture (2 min)

Réponds dans ta tête, puis déplie la réponse. Ce sont les questions qu'on se pose à voix haute au début de la séance.

**1. Pourquoi transformer « male » / « female » en 0 / 1 ?**
- a. Un modèle ne comprend que des nombres
- b. Pour économiser de la mémoire
- c. C'est une règle de Kaggle
- d. Pour anonymiser les passagers

<details><summary>Réponse</summary>

**a. Un modèle ne comprend que des nombres** — scikit-learn fait des calculs. Un mot n'est pas un nombre, il faut l'encoder.

</details>

**2. Quel fichier dépose-t-on sur Kaggle pour être classé ?**
- a. Le notebook
- b. train.csv
- c. submission.csv, avec une prédiction par passager de test.csv
- d. Une capture d'écran du score

<details><summary>Réponse</summary>

**c. submission.csv, avec une prédiction par passager de test.csv** — Deux colonnes : PassengerId et Survived. Kaggle compare avec les vraies réponses qu'il garde secrètes.

</details>


## Préparation

Mêmes données que la leçon : `train.csv` de Kaggle s'il est dans Colab, sinon une copie publique. On recharge aussi `regrouper_titre()` et `preparer()`, plus deux petites fonctions : `verifier()` (✅ / ❌) et `proche()` (compare deux nombres avec une tolérance).

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

URL_SECOURS = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

if os.path.exists("train.csv"):
    train = pd.read_csv("train.csv")
    print("train.csv de Kaggle chargé")
else:
    try:
        train = pd.read_csv(URL_SECOURS)
        print("Copie publique de train.csv chargée (mêmes colonnes que Kaggle)")
    except Exception as erreur:
        print("Impossible de charger les données : pas de réseau ?", erreur)
        raise
print(train.shape[0], "passagers,", train.shape[1], "colonnes")

# --- La recette de la leçon : regrouper_titre() et preparer() ---
def regrouper_titre(titre):
    """Garde les 4 titres fréquents, regroupe le reste dans « Autre »."""
    if titre in ["Mr", "Mrs", "Miss", "Master"]:
        return titre
    if titre in ["Mlle", "Ms"]:
        return "Miss"
    if titre == "Mme":
        return "Mrs"
    return "Autre"          # Dr, Rev, Col, Major, Countess, Capt...


COLONNES = ["Pclass", "Sex", "Age", "Fare", "Embarked", "Famille", "Seul", "Titre"]

def preparer(df):
    """Transforme le tableau brut de Kaggle en tableau de nombres prêt pour un modèle."""
    d = df.copy()
    # 1. Nouvelles variables
    d["Famille"] = d["SibSp"] + d["Parch"] + 1
    d["Seul"] = (d["Famille"] == 1).astype(int)
    d["Titre"] = d["Name"].str.extract(r",\s*([^\.]+)\.")[0].str.strip().apply(regrouper_titre)
    # 2. Cases vides : l'âge médian de chaque titre (un « Master » a 4 ans, un « Mr » 30)
    d["Age"] = d.groupby("Titre")["Age"].transform(lambda s: s.fillna(s.median()))
    d["Age"] = d["Age"].fillna(d["Age"].median())
    d["Fare"] = d["Fare"].fillna(d["Fare"].median())
    d["Embarked"] = d["Embarked"].fillna("S")
    # 3. Tout en nombres
    d["Sex"] = (d["Sex"] == "female").astype(int)
    d["Embarked"] = d["Embarked"].map({"S": 0, "C": 1, "Q": 2})
    d["Titre"] = d["Titre"].map({"Mr": 0, "Mrs": 1, "Miss": 2, "Master": 3, "Autre": 4})
    return d[COLONNES]

# --- Vérification automatique : affiche ✅ ou ❌, ne plante jamais ---
def verifier(nom, condition):
    """condition : un booléen, ou une fonction sans argument qui renvoie un booléen."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))


def proche(valeur, attendu, tolerance=0.01):
    """Vrai si valeur est un nombre à moins de `tolerance` de attendu."""
    try:
        return abs(float(valeur) - attendu) <= tolerance
    except (TypeError, ValueError):
        return False

print("Prêt ! Données, preparer() et verifier() sont chargés.")

## Exercice 1 ⭐ · Premier coup d'œil

Avant de modéliser, on regarde le tableau `train`. Remplis les 4 variables :
- `nb_passagers` : nombre de lignes
- `nb_colonnes` : nombre de colonnes
- `nb_ages_manquants` : nombre de cases vides dans la colonne `Age`
- `colonne_la_plus_vide` : le nom de la colonne qui a le plus de cases vides (une chaîne de caractères)

Résultat attendu : `891 passagers, 12 colonnes, 177 âges manquants`.

<details><summary>Indice</summary>

`train.shape` donne (lignes, colonnes). `train.isna().sum()` compte les vides par colonne, et `.idxmax()` donne le nom de la plus grande valeur.
</details>

In [ ]:
# À toi
nb_passagers = None
nb_colonnes = None
nb_ages_manquants = None
colonne_la_plus_vide = None

print(nb_passagers, "passagers,", nb_colonnes, "colonnes,", nb_ages_manquants, "âges manquants, colonne la plus vide :", colonne_la_plus_vide)

In [ ]:
verifier("Exercice 1 · nb_passagers", nb_passagers == 891)
verifier("Exercice 1 · nb_colonnes", nb_colonnes == 12)
verifier("Exercice 1 · nb_ages_manquants", nb_ages_manquants == 177)
verifier("Exercice 1 · colonne_la_plus_vide", colonne_la_plus_vide == "Cabin")

<details><summary>Solution</summary>

```python
nb_passagers = train.shape[0]
nb_colonnes = train.shape[1]
vides = train.isna().sum()
nb_ages_manquants = vides["Age"]
colonne_la_plus_vide = vides.idxmax()      # Cabin : 687 cases vides sur 891

print(nb_passagers, "passagers,", nb_colonnes, "colonnes,", nb_ages_manquants, "âges manquants, colonne la plus vide :", colonne_la_plus_vide)
```
</details>

## Exercice 2 ⭐ · Compter avec value_counts

Avec `value_counts()` et `mean()`, trouve :
- `nb_troisieme_classe` : combien de passagers voyagent en 3e classe (`Pclass == 3`)
- `port_principal` : la lettre du port d'embarquement le plus fréquent (`"S"`, `"C"` ou `"Q"`)
- `part_survivants` : la proportion de survivants dans `train` (un nombre entre 0 et 1)

Résultat attendu : `491` en 3e classe, port `S`, et environ `0.38` de survivants.

<details><summary>Indice</summary>

`train["Pclass"].value_counts()` renvoie une Series : on lit la valeur pour la clé 3 avec `[3]`. Pour le port, `.idxmax()` ou `.mode()[0]`. La moyenne d'une colonne de 0 et de 1 est une proportion.
</details>

In [ ]:
# À toi
nb_troisieme_classe = None
port_principal = None
part_survivants = None

print(nb_troisieme_classe, port_principal, part_survivants)

In [ ]:
verifier("Exercice 2 · nb_troisieme_classe", nb_troisieme_classe == 491)
verifier("Exercice 2 · port_principal", port_principal == "S")
verifier("Exercice 2 · part_survivants", proche(part_survivants, 0.384, 0.005))

<details><summary>Solution</summary>

```python
nb_troisieme_classe = train["Pclass"].value_counts()[3]
port_principal = train["Embarked"].value_counts().idxmax()     # ou train["Embarked"].mode()[0]
part_survivants = train["Survived"].mean()

print(nb_troisieme_classe, port_principal, round(part_survivants, 3))
```
</details>

## Exercice 3 ⭐ · Hypothèse : « les femmes et les enfants d'abord »

Vérifie la règle du Titanic avec `groupby`. Calcule :
- `taux_femmes` : taux de survie des femmes (`Sex == "female"`)
- `taux_hommes` : taux de survie des hommes
- `taux_enfants` : taux de survie des passagers de moins de 12 ans (`Age < 12`)

Résultat attendu : environ `0.74` pour les femmes, `0.19` pour les hommes, `0.57` pour les enfants.

<details><summary>Indice</summary>

`train.groupby("Sex")["Survived"].mean()` donne une Series avec les clés `female` et `male`. Pour les enfants, filtre d'abord : `train[train["Age"] < 12]`.
</details>

In [ ]:
# À toi
survie_par_sexe = None          # Series : taux de survie par sexe
taux_femmes = None
taux_hommes = None
taux_enfants = None

print(survie_par_sexe)
print("Enfants :", taux_enfants)

In [ ]:
verifier("Exercice 3 · taux_femmes", proche(taux_femmes, 0.742, 0.005))
verifier("Exercice 3 · taux_hommes", proche(taux_hommes, 0.189, 0.005))
verifier("Exercice 3 · taux_enfants", proche(taux_enfants, 0.574, 0.005))

<details><summary>Solution</summary>

```python
survie_par_sexe = train.groupby("Sex")["Survived"].mean()
taux_femmes = survie_par_sexe["female"]
taux_hommes = survie_par_sexe["male"]
taux_enfants = train[train["Age"] < 12]["Survived"].mean()

print(survie_par_sexe.round(2))
print("Enfants :", round(taux_enfants, 2))
# 74 % des femmes survivent contre 19 % des hommes : la règle a bien été appliquée.
```
</details>

## Exercice 4 ⭐ · Hypothèse : la classe compte

Même méthode pour la classe du billet :
- `survie_par_classe` : Series du taux de survie pour chaque valeur de `Pclass` (1, 2, 3)
- `classe_la_mieux_lotie` : le numéro de la classe qui survit le mieux (un entier)
- `ecart` : différence entre le taux de la 1re classe et celui de la 3e

Puis trace un graphique en barres de `survie_par_classe`. Résultat attendu : `0.63`, `0.47`, `0.24` et un écart d'environ `0.39`.

<details><summary>Indice</summary>

`groupby("Pclass")["Survived"].mean()` puis `.idxmax()` pour la meilleure classe. Pour le graphique : `survie_par_classe.plot(kind="bar")` puis `plt.show()`.
</details>

In [ ]:
# À toi
survie_par_classe = None
classe_la_mieux_lotie = None
ecart = None

print(survie_par_classe)
print("Écart 1re - 3e :", ecart)

In [ ]:
verifier("Exercice 4 · survie_par_classe (3 classes)", lambda: len(survie_par_classe) == 3 and proche(survie_par_classe[1], 0.63, 0.005))
verifier("Exercice 4 · classe_la_mieux_lotie", classe_la_mieux_lotie == 1)
verifier("Exercice 4 · ecart", proche(ecart, 0.388, 0.005))

<details><summary>Solution</summary>

```python
survie_par_classe = train.groupby("Pclass")["Survived"].mean()
classe_la_mieux_lotie = survie_par_classe.idxmax()
ecart = survie_par_classe[1] - survie_par_classe[3]

print(survie_par_classe.round(2))
print("Écart 1re - 3e :", round(ecart, 2))

survie_par_classe.plot(kind="bar", figsize=(5, 3.5), color="tab:orange")
plt.title("Taux de survie par classe")
plt.xticks(rotation=0)
plt.show()
```
</details>

## Exercice 5 ⭐⭐ · Nettoyer les cases vides, plus finement

Dans la leçon, on a rempli `Age` avec la médiane globale (28 ans) et obtenu un vilain pic. Fais mieux : remplis chaque âge manquant avec **la médiane de sa classe** (37 ans en 1re, 29 en 2e, 24 en 3e).
- `age_rempli` : Series de 891 âges, sans aucune case vide, où les manquants ont reçu la médiane de leur classe
- `embarked_rempli` : Series `Embarked` où les 2 vides ont reçu le port le plus fréquent

Résultat attendu : `age_rempli` a une moyenne d'environ `29.07` ans et une médiane de `26.0` ; `embarked_rempli` compte `646` fois `S`.

<details><summary>Indice</summary>

`train.groupby("Pclass")["Age"].transform(lambda s: s.fillna(s.median()))` applique la médiane de chaque groupe à ses propres vides. C'est la même astuce que `preparer()` avec le titre.
</details>

In [ ]:
# À toi
age_rempli = None
embarked_rempli = None

print("Vides restants dans age_rempli :", None)

In [ ]:
verifier("Exercice 5 · age_rempli sans vide", lambda: len(age_rempli) == 891 and age_rempli.isna().sum() == 0)
verifier("Exercice 5 · médiane par classe (moyenne ≈ 29.07)", lambda: proche(age_rempli.mean(), 29.066, 0.01) and age_rempli.median() == 26.0)
verifier("Exercice 5 · embarked_rempli", lambda: embarked_rempli.isna().sum() == 0 and embarked_rempli.value_counts()["S"] == 646)

<details><summary>Solution</summary>

```python
age_rempli = train.groupby("Pclass")["Age"].transform(lambda s: s.fillna(s.median()))
embarked_rempli = train["Embarked"].fillna(train["Embarked"].mode()[0])

print("Vides restants dans age_rempli :", age_rempli.isna().sum())
print("Moyenne :", round(age_rempli.mean(), 2), "- médiane :", age_rempli.median())

# Comparaison : le pic à 28 ans de la leçon est réparti sur 3 valeurs
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
axes[0].hist(train["Age"].fillna(train["Age"].median()), bins=30, color="tab:red")
axes[0].set_title("Médiane globale : un pic à 28 ans")
axes[1].hist(age_rempli, bins=30, color="tab:green")
axes[1].set_title("Médiane par classe : 3 petits pics")
plt.show()
```
</details>

## Exercice 6 ⭐⭐ · Le titre caché dans le nom (regex)

Un nom ressemble à `Braund, Mr. Owen Harris` : le titre est entre la virgule et le point. Avec `str.extract` et l'expression régulière de la leçon :
- `titres` : Series des titres bruts (`Mr`, `Mrs`, `Miss`, `Master`, `Dr`, `Rev`...), sans espace autour
- `titres_regroupes` : la même Series après `regrouper_titre` (5 valeurs possibles)
- `age_median_master` : l'âge médian des passagers dont le titre brut est `Master`

Résultat attendu : `17` titres différents, `40` Master, `23` « Autre » après regroupement, et `age_median_master = 3.5`.

<details><summary>Indice</summary>

`train["Name"].str.extract(r",\s*([^\.]+)\.")[0].str.strip()` puis `.apply(regrouper_titre)`. Pour l'âge médian : `train["Age"][titres == "Master"].median()`.
</details>

In [ ]:
# À toi
titres = None
titres_regroupes = None
age_median_master = None

print(None)                       # affiche titres.value_counts()
print("Âge médian d'un Master :", age_median_master)

In [ ]:
verifier("Exercice 6 · 17 titres bruts, 40 Master", lambda: titres.nunique() == 17 and (titres == "Master").sum() == 40)
verifier("Exercice 6 · titres_regroupes (5 valeurs, 23 Autre)", lambda: titres_regroupes.nunique() == 5 and (titres_regroupes == "Autre").sum() == 23)
verifier("Exercice 6 · age_median_master", proche(age_median_master, 3.5, 0.01))

<details><summary>Solution</summary>

```python
titres = train["Name"].str.extract(r",\s*([^\.]+)\.")[0].str.strip()
titres_regroupes = titres.apply(regrouper_titre)
age_median_master = train["Age"][titres == "Master"].median()

print(titres.value_counts())
print(titres_regroupes.value_counts())
print("Âge médian d'un Master :", age_median_master)   # 3,5 ans : « Master » = petit garçon
```
</details>

## Exercice 7 ⭐⭐ · Trois nouvelles variables

Ajoute 3 colonnes à `train` (feature engineering) :
- `Famille` = `SibSp` + `Parch` + 1
- `Seul` = 1 si `Famille == 1`, sinon 0 (un entier)
- `Enfant` = 1 si `Age < 12`, sinon 0

Puis calcule `taux_survie_seuls` (taux de survie quand `Seul == 1`), `taux_survie_accompagnes` (`Seul == 0`) et `nb_enfants`.
Résultat attendu : `0.30` contre `0.51`, et `68` enfants. La plus grande famille à bord compte `11` personnes.

<details><summary>Indice</summary>

`(condition).astype(int)` transforme Vrai/Faux en 1/0. Pour les taux : `train.groupby("Seul")["Survived"].mean()` puis `[1]` et `[0]`.
</details>

In [ ]:
# À toi
train["Famille"] = None
train["Seul"] = None
train["Enfant"] = None

taux_survie_seuls = None
taux_survie_accompagnes = None
nb_enfants = None

print("Seuls :", taux_survie_seuls, "- accompagnés :", taux_survie_accompagnes, "- enfants :", nb_enfants)

In [ ]:
verifier("Exercice 7 · Famille (max 11)", lambda: train["Famille"].max() == 11 and train["Famille"].min() == 1)
verifier("Exercice 7 · Seul (537 passagers seuls, en 0/1)", lambda: train["Seul"].sum() == 537 and set(train["Seul"].unique()) == {0, 1})
verifier("Exercice 7 · Enfant", lambda: train["Enfant"].sum() == 68 and nb_enfants == 68)
verifier("Exercice 7 · taux de survie", proche(taux_survie_seuls, 0.304, 0.005) and proche(taux_survie_accompagnes, 0.506, 0.005))

<details><summary>Solution</summary>

```python
train["Famille"] = train["SibSp"] + train["Parch"] + 1
train["Seul"] = (train["Famille"] == 1).astype(int)
train["Enfant"] = (train["Age"] < 12).astype(int)

survie_seul = train.groupby("Seul")["Survived"].mean()
taux_survie_seuls = survie_seul[1]
taux_survie_accompagnes = survie_seul[0]
nb_enfants = train["Enfant"].sum()

print("Seuls :", round(taux_survie_seuls, 2), "- accompagnés :", round(taux_survie_accompagnes, 2), "- enfants :", nb_enfants)
```
</details>

## Exercice 8 ⭐⭐ · Ta version de preparer()

`preparer()` (chargée dans la Préparation) renvoie 8 colonnes. Écris `preparer_v2(df)` qui fait la même chose **plus** 2 colonnes :
- `Prix_par_personne` = `Fare` / `Famille` (un billet à 100 livres pour 4 personnes, c'est 25 livres chacun)
- `Cabine_connue` = 1 si `Cabin` n'est pas vide, sinon 0

Puis `X2 = preparer_v2(train)`. Résultat attendu : `X2.shape == (891, 10)`, aucune case vide, moyenne de `Prix_par_personne` ≈ `19.92`, et `204` cabines connues.

<details><summary>Indice</summary>

Le plus simple : `d = preparer(df)` puis ajoute les 2 colonnes à `d` à partir de `df` (attention, `Famille` est déjà dans `d`, et `d["Fare"]` n'a plus de vide). Pense à `.values` si les index diffèrent, ou travaille avec `df.copy()` comme dans la leçon.
</details>

In [ ]:
# À toi
def preparer_v2(df):
    d = preparer(df)
    # ajoute Prix_par_personne et Cabine_connue à d
    return d


X2 = preparer_v2(train)
y = train["Survived"]
print(X2.shape)
X2.head(3)

In [ ]:
verifier("Exercice 8 · 10 colonnes (les 8 de la leçon + 2), 0 vide", lambda: X2.shape == (891, 10) and X2.isna().sum().sum() == 0 and all(c in X2.columns for c in COLONNES))
verifier("Exercice 8 · Prix_par_personne", lambda: proche(X2["Prix_par_personne"].mean(), 19.916, 0.01))
verifier("Exercice 8 · Cabine_connue", lambda: X2["Cabine_connue"].sum() == 204)

<details><summary>Solution</summary>

```python
def preparer_v2(df):
    d = preparer(df)
    d["Prix_par_personne"] = d["Fare"] / d["Famille"]
    d["Cabine_connue"] = df["Cabin"].notna().astype(int).values
    return d


X2 = preparer_v2(train)
y = train["Survived"]
print(X2.shape)
X2.head(3)
```
</details>

## Exercice 9 ⭐⭐ · Régression logistique sur 25 % cachés

On mesure un premier modèle honnêtement : on cache 25 % des passagers.
1. `X = preparer(train)` et `y = train["Survived"]`
2. `train_test_split(X, y, test_size=0.25, random_state=42)` → `X_train, X_test, y_train, y_test`
3. Entraîne `logistique = LogisticRegression(max_iter=1000)` sur la partie train
4. `exactitude_logistique` : exactitude sur `X_test` (méthode `.score`)

Résultat attendu : `223` passagers cachés et une exactitude autour de `0.79`.

<details><summary>Indice</summary>

La méthode `.fit(X_train, y_train)` apprend, `.score(X_test, y_test)` mesure la proportion de bonnes réponses.
</details>

In [ ]:
# À toi
X = preparer(train)
y = train["Survived"]

X_train, X_test, y_train, y_test = None, None, None, None
logistique = None
exactitude_logistique = None

print("Exactitude :", exactitude_logistique)

In [ ]:
verifier("Exercice 9 · découpage 25 % (223 passagers cachés)", lambda: len(X_test) == 223 and len(X_train) == 668)
verifier("Exercice 9 · logistique entraînée", lambda: hasattr(logistique, "coef_"))
verifier("Exercice 9 · exactitude_logistique ≈ 0.79", proche(exactitude_logistique, 0.794, 0.02))

<details><summary>Solution</summary>

```python
X = preparer(train)
y = train["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
logistique = LogisticRegression(max_iter=1000)
logistique.fit(X_train, y_train)
exactitude_logistique = logistique.score(X_test, y_test)

print("Exactitude :", round(exactitude_logistique, 3))
```
</details>

## Exercice 10 ⭐⭐⭐ · Forêt aléatoire et validation croisée

Un seul découpage, c'est un seul contrôle blanc. Passe à la **validation croisée** en 5 paquets :
- `scores_logistique` : `cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5)`
- `scores_foret` : la même chose avec `RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)`
- `meilleur_modele` : `"forêt"` ou `"logistique"`, celui dont la **moyenne** des 5 scores est la plus haute

Résultat attendu : 5 scores par modèle, logistique ≈ `0.80` en moyenne, forêt ≈ `0.83`.

<details><summary>Indice</summary>

`cross_val_score` renvoie un tableau numpy de 5 exactitudes : `.mean()` pour la moyenne, `.std()` pour la dispersion. Compare les deux moyennes.
</details>

In [ ]:
# À toi
scores_logistique = None
scores_foret = None
meilleur_modele = None

print("Logistique :", scores_logistique)
print("Forêt      :", scores_foret)
print("Meilleur   :", meilleur_modele)

In [ ]:
verifier("Exercice 10 · 5 scores par modèle", lambda: len(scores_logistique) == 5 and len(scores_foret) == 5)
verifier("Exercice 10 · logistique ≈ 0.80", lambda: proche(np.mean(scores_logistique), 0.8025, 0.02))
verifier("Exercice 10 · forêt ≈ 0.83", lambda: proche(np.mean(scores_foret), 0.827, 0.02))
verifier("Exercice 10 · meilleur_modele", lambda: meilleur_modele.lower().startswith("for"))

<details><summary>Solution</summary>

```python
scores_logistique = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5)
foret = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
scores_foret = cross_val_score(foret, X, y, cv=5)
meilleur_modele = "forêt" if scores_foret.mean() > scores_logistique.mean() else "logistique"

print(f"Logistique : {scores_logistique.mean()*100:.1f} % (± {scores_logistique.std()*100:.1f})")
print(f"Forêt      : {scores_foret.mean()*100:.1f} % (± {scores_foret.std()*100:.1f})")
print("Meilleur   :", meilleur_modele)
```
</details>

## Exercice 11 ⭐⭐⭐ · Un faux test.csv et ta soumission

Tu n'as peut-être pas `test.csv` sous la main : on en fabrique un. La cellule met de côté 100 passagers de `train` (`faux_test`, **sans** la colonne `Survived`) et garde leurs vraies réponses dans `reponses_cachees`. Les 791 autres forment `train_reduit`.

À toi de faire exactement ce qu'on fera sur Kaggle :
1. Entraîne une forêt (`n_estimators=200, max_depth=5, random_state=42`) sur `preparer(train_reduit)` et `train_reduit["Survived"]`
2. Prédis `preparer(faux_test)` → `predictions`
3. Construis `submission` : un DataFrame à 2 colonnes `PassengerId` et `Survived`, et écris-le dans `submission.csv` (`index=False`)
4. `exactitude_faux_test` : proportion de prédictions égales à `reponses_cachees`

Résultat attendu : un fichier de 100 lignes et 2 colonnes, une exactitude d'environ `0.77`.

<details><summary>Indice</summary>

La recette de la leçon, section 8 : `modele.fit(...)`, `modele.predict(preparer(faux_test))`, `pd.DataFrame({"PassengerId": faux_test["PassengerId"], "Survived": predictions}).to_csv("submission.csv", index=False)`. Pour l'exactitude : `(predictions == reponses_cachees.values).mean()`.
</details>

In [ ]:
faux_test = train.sample(100, random_state=1)
reponses_cachees = faux_test["Survived"]           # on triche un peu : on garde les vraies réponses pour se noter
faux_test = faux_test.drop(columns=["Survived"])   # comme test.csv : pas de colonne Survived
train_reduit = train.drop(faux_test.index)
print(len(train_reduit), "passagers pour apprendre,", len(faux_test), "à prédire")

# À toi
modele = None
predictions = None
submission = None
exactitude_faux_test = None

print("Exactitude sur le faux test :", exactitude_faux_test)

In [ ]:
verifier("Exercice 11 · submission (DataFrame) et submission.csv existent", lambda: isinstance(submission, pd.DataFrame) and os.path.exists("submission.csv"))
verifier("Exercice 11 · 100 lignes, colonnes PassengerId et Survived",
         lambda: pd.read_csv("submission.csv").shape == (100, 2) and list(pd.read_csv("submission.csv").columns) == ["PassengerId", "Survived"])
verifier("Exercice 11 · prédictions en 0/1", lambda: set(pd.read_csv("submission.csv")["Survived"].unique()) <= {0, 1})
verifier("Exercice 11 · exactitude_faux_test ≈ 0.77", proche(exactitude_faux_test, 0.77, 0.04))

<details><summary>Solution</summary>

```python
faux_test = train.sample(100, random_state=1)
reponses_cachees = faux_test["Survived"]
faux_test = faux_test.drop(columns=["Survived"])
train_reduit = train.drop(faux_test.index)
print(len(train_reduit), "passagers pour apprendre,", len(faux_test), "à prédire")

modele = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
modele.fit(preparer(train_reduit), train_reduit["Survived"])
predictions = modele.predict(preparer(faux_test))

submission = pd.DataFrame({"PassengerId": faux_test["PassengerId"], "Survived": predictions})
submission.to_csv("submission.csv", index=False)
exactitude_faux_test = (predictions == reponses_cachees.values).mean()

print(submission.head())
print("Exactitude sur le faux test :", exactitude_faux_test)
# Sur Kaggle, c'est pareil avec le vrai test.csv : tu ne connais juste pas reponses_cachees.
```
</details>

## Exercice 12 ⭐⭐⭐ · Défi · Rose et Jack auraient-ils survécu ?

Combine tout : écris une fonction `predire_survie(nom, sexe, classe, age, prix, sibsp=0, parch=0, port="S")` qui
1. construit un DataFrame d'**une ligne** avec les colonnes brutes de Kaggle (`PassengerId`, `Pclass`, `Name`, `Sex`, `Age`, `SibSp`, `Parch`, `Ticket`, `Fare`, `Cabin`, `Embarked`),
2. le passe dans `preparer()` (le titre sera lu dans `nom`, donc écris-le au format `"Nom, Titre. Prénom"`),
3. renvoie la **probabilité de survie** (un nombre entre 0 et 1) donnée par `modele_complet`, une forêt (`200` arbres, `max_depth=5`, `random_state=42`) entraînée sur **tout** `train`.

Teste avec Rose (`"DeWitt Bukater, Miss. Rose"`, femme, 1re classe, 17 ans, 150 livres, `parch=1`) et Jack (`"Dawson, Mr. Jack"`, homme, 3e classe, 20 ans, 7,5 livres).
Résultat attendu : `proba_rose` > 0.7 et `proba_jack` < 0.3.

<details><summary>Indice</summary>

`pd.DataFrame([{"PassengerId": 0, "Pclass": classe, "Name": nom, ...}])` crée une ligne. `modele_complet.predict_proba(ligne_preparee)[0][1]` est la probabilité de la classe 1 (survit). `Cabin` peut valoir `None`, `Ticket` n'importe quoi.
</details>

In [ ]:
modele_complet = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
modele_complet.fit(preparer(train), train["Survived"])

# À toi
def predire_survie(nom, sexe, classe, age, prix, sibsp=0, parch=0, port="S"):
    """sexe : "female" ou "male" ; port : "S", "C" ou "Q". Renvoie une probabilité de survie."""
    return None


proba_rose = predire_survie("DeWitt Bukater, Miss. Rose", "female", 1, 17, 150.0, parch=1)
proba_jack = predire_survie("Dawson, Mr. Jack", "male", 3, 20, 7.5)
print("Rose :", proba_rose, "- Jack :", proba_jack)

In [ ]:
verifier("Exercice 12 · la fonction renvoie une probabilité", lambda: 0 <= float(proba_rose) <= 1 and 0 <= float(proba_jack) <= 1)
verifier("Exercice 12 · Rose survit (> 0.7)", lambda: float(proba_rose) > 0.7)
verifier("Exercice 12 · Jack ne survit pas (< 0.3)", lambda: float(proba_jack) < 0.3)
verifier("Exercice 12 · un Master de 4 ans en 2e classe survit (> 0.5)", lambda: float(predire_survie("Petit, Master. Louis", "male", 2, 4, 30.0, parch=2)) > 0.5)

<details><summary>Solution</summary>

```python
modele_complet = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
modele_complet.fit(preparer(train), train["Survived"])


def predire_survie(nom, sexe, classe, age, prix, sibsp=0, parch=0, port="S"):
    """sexe : "female" ou "male" ; port : "S", "C" ou "Q". Renvoie une probabilité de survie."""
    passager = pd.DataFrame([{
        "PassengerId": 0, "Pclass": classe, "Name": nom, "Sex": sexe, "Age": age,
        "SibSp": sibsp, "Parch": parch, "Ticket": "?", "Fare": prix, "Cabin": None, "Embarked": port,
    }])
    return modele_complet.predict_proba(preparer(passager))[0][1]


proba_rose = predire_survie("DeWitt Bukater, Miss. Rose", "female", 1, 17, 150.0, parch=1)
proba_jack = predire_survie("Dawson, Mr. Jack", "male", 3, 20, 7.5)
print(f"Rose : {proba_rose*100:.0f} % - Jack : {proba_jack*100:.0f} %")
# Le film avait raison : le modèle donne plus de 90 % à Rose et environ 10 % à Jack.
```
</details>

## Bravo !

Tu as refait seul·e tout le pipeline de la séance : explorer, vérifier des hypothèses, nettoyer, créer des variables, entraîner, mesurer honnêtement et produire un fichier de soumission.

**Pour aller plus loin** : remplace `faux_test` par le vrai `test.csv` de Kaggle dans l'exercice 11, soumets `submission.csv` sur https://www.kaggle.com/competitions/titanic et note ton score. Puis essaie `preparer_v2` à la place de `preparer` : le score bouge-t-il ?